# KolektorSDD2 exploratory data analysis

This notebook counts images and ground-truth masks by class, reports image sizes, shows two examples per class, and preserves the official test split while creating a stratified 80% train / 20% validation split from the official training data.

The split is kept in memory and does not move or modify the DVC-tracked files.


In [1]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
DATA_ROOT = REPO_ROOT / "data" / "raw" / "kolektor-sdd2"
SEED = 42

records = []
excluded_files = []

for source_split in ["train", "test"]:
    for image_path in sorted((DATA_ROOT / source_split).glob("*.png")):
        if "_GT" in image_path.stem or "(copy)" in image_path.stem:
            excluded_files.append(image_path)
            continue

        mask_path = image_path.with_name(f"{image_path.stem}_GT.png")
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) if mask_path.exists() else None

        if image is None:
            raise ValueError(f"Could not read {image_path}")

        height, width = image.shape[:2]
        defect_pixels = int(np.count_nonzero(mask)) if mask is not None else 0

        records.append({
            "source_split": source_split,
            "image_path": image_path,
            "mask_path": mask_path if mask_path.exists() else None,
            "mask_exists": mask is not None,
            "height": height,
            "width": width,
            "channels": image.shape[2],
            "defect_pixels": defect_pixels,
            "class": "defect" if defect_pixels > 0 else "non-defect",
        })

df = pd.DataFrame(records)

print("Analyzed images:", len(df))
print("Excluded archive files:", len(excluded_files))
print("Missing masks:", int((~df["mask_exists"]).sum()))

print("Images and masks by class")
display(
    df.groupby("class")
      .agg(
          images=("image_path", "size"),
          masks=("mask_exists", "sum"),
          total_defect_pixels=("defect_pixels", "sum"),
          median_defect_pixels=("defect_pixels", "median"),
      )
      .reindex(["defect", "non-defect"])
)

print("Images and masks by original dataset split")
display(
    df.groupby(["source_split", "class"])
      .agg(images=("image_path", "size"), masks=("mask_exists", "sum"))
)

print("Most common image sizes")
size_counts = (
    df.groupby(["height", "width"], as_index=False)
      .size()
      .sort_values("size", ascending=False)
      .rename(columns={"size": "images"})
)
display(size_counts.head(15))
display(df[["height", "width", "channels"]].describe())

def show_examples(class_name, n=2):
    examples = df[df["class"] == class_name].sort_values("image_path").head(n)
    figure, axes = plt.subplots(len(examples), 3, figsize=(15, 4 * len(examples)))
    axes = np.atleast_2d(axes)

    for row_index, (_, sample) in enumerate(examples.iterrows()):
        image = cv2.cvtColor(cv2.imread(str(sample["image_path"])), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(sample["mask_path"]), cv2.IMREAD_GRAYSCALE)

        axes[row_index, 0].imshow(image)
        axes[row_index, 0].set_title(class_name + ": " + sample["image_path"].name)
        axes[row_index, 0].axis("off")

        axes[row_index, 1].imshow(mask, cmap="gray", vmin=0, vmax=255)
        axes[row_index, 1].set_title("ground-truth mask")
        axes[row_index, 1].axis("off")

        axes[row_index, 2].imshow(image)
        axes[row_index, 2].imshow(mask > 0, cmap="Reds", alpha=0.45)
        axes[row_index, 2].set_title("image plus mask overlay")
        axes[row_index, 2].axis("off")

    figure.tight_layout()
    plt.show()

show_examples("defect")
show_examples("non-defect")

RATIOS = {"train": 0.80, "validation": 0.20}

def allocate_counts(n, ratios):
    names = list(ratios)
    raw = np.array([ratios[name] * n for name in names])
    counts = np.floor(raw).astype(int)
    remainder = n - counts.sum()

    for index in np.argsort(-(raw - counts))[:remainder]:
        counts[index] += 1

    return dict(zip(names, counts))

def make_official_split(frame, seed=42):
    result = frame.copy()
    result["dataset_split"] = np.where(result["source_split"] == "test", "test", None)
    random_generator = np.random.default_rng(seed)

    for class_name in sorted(result["class"].unique()):
        indices = result.index[(result["source_split"] == "train") & (result["class"] == class_name)].to_numpy()
        indices = random_generator.permutation(indices)
        counts = allocate_counts(len(indices), RATIOS)

        start = 0
        for split_name in RATIOS:
            stop = start + counts[split_name]
            result.loc[indices[start:stop], "dataset_split"] = split_name
            start = stop

    return result

split_df = make_official_split(df, seed=SEED)

if split_df["dataset_split"].isna().any():
    raise ValueError("Some images were not assigned to a split.")

print("Official test preserved; official train split 80/20 into train and validation")
display(
    split_df.groupby(["dataset_split", "class"])
            .size()
            .unstack(fill_value=0)
            .reindex(["train", "validation", "test"])
)
display(split_df["dataset_split"].value_counts().reindex(["train", "validation", "test"]))

manifest = split_df.copy()
manifest["image_path"] = manifest["image_path"].map(lambda path: Path(path).relative_to(REPO_ROOT).as_posix())
manifest["mask_path"] = manifest["mask_path"].map(lambda path: Path(path).relative_to(REPO_ROOT).as_posix() if path is not None else None)
manifest_path = REPO_ROOT / "data" / "processed" / "splits.csv"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest.to_csv(manifest_path, index=False)
print("Saved reusable split manifest:", manifest_path)


Analyzed images: 3335
Excluded archive files: 3337
Missing masks: 0
Images and masks by class


,images,masks,total_defect_pixels,median_defect_pixels
class,,,,
defect,356,356,1451751,2273.0
non-defect,2979,2979,0,0.0


Images and masks by original dataset split


images  masks
source_split class                    
test         defect         110    110
             non-defect     894    894
train        defect         246    246
             non-defect    2085   2085

Most common image sizes


...

C:\Users\carlo\AppData\Local\Temp\ipykernel_62904\3612789628.py:102: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Official test preserved; official train split 80/20 into train and validation


class,defect,non-defect
dataset_split,,
train,197,1668
validation,49,417
test,110,894


dataset_split
train         1865
validation     466
test          1004
Name: count, dtype: int64

Saved reusable split manifest: C:\Users\carlo\Documents\CodexWorkspace\Study_Github\data\processed\splits.csv


## Interpretation

A defect sample has at least one nonzero pixel in its ground-truth mask. A non-defect sample has an all-black mask.

The split is deterministic because it uses seed 42. The reusable manifest is saved at data/processed/splits.csv with repository-relative paths.
